# SPARQL Queries as RDF — User Guide

`starsparql` represents a SPARQL query itself as an RDF graph, based on SPARQL 1.2 algebra, using the `salg:` ontology (specific to starlayer). Queries can be exported to RDF and imported back. A corresponding SHACL shapes graph validates the encoding; `starshacl` separately extends the SHACL meta-shape graph for SHACL 1.2 semantics.

See the [SPARQL guide](03-sparql.ipynb) for ordinary query semantics and built-in functions - this guide is specifically about treating a *query itself* as data you can encode, inspect, edit, and validate, a separate concern from writing or running a query.

Grounded in one tiny graph and one tiny query throughout, so each step's effect shows up as a real, different query result — not just a triple count. Eight steps: build a graph + query → parse the query into a `Query` object → encode its algebra as RDF → edit the query by editing the RDF → decode back into a real `Query` and run it → export the edited query back to text → load the ontology/shapes graphs → validate an encoded query with RDFS reasoning.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells from top to bottom — later steps reuse variables from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace

EX = Namespace("http://example.org/")

## Step 1 — a graph and a query

Everything below is grounded in this one tiny graph and one tiny query, so later steps' effects show up as real, different query results — not just triple counts.

In [2]:
import starsparql

g = StarLayerGraph()
g.bind("ex", EX)
g.add((EX.bob, EX.knows, EX.carol))
g.add((EX.bob, EX.likes, EX.dana))

text_query = """PREFIX ex: <http://example.org/>
SELECT ?s ?o WHERE { ?s ex:knows ?o }"""

for row in g.query(text_query):
    print(g.qname(row.s), g.qname(row.o))

ex:bob ex:carol


## Step 2 — parse the query text into a `Query` object

`prepare_query_12()` parses text into a real, executable-shaped `rdflib.Query` — the SPARQL-1.2-aware counterpart to plain rdflib's own `prepareQuery()`. Its grammar is a strict superset of SPARQL 1.1, so it's the right function to call here even though this particular query uses no RDF 1.2 syntax at all.

In [3]:
parsed = starsparql.prepare_query_12(text_query)

print(type(parsed))
print("top-level algebra op:", parsed.algebra.name)

<class 'rdflib.plugins.sparql.sparql.Query'>
top-level algebra op: SelectQuery


## Step 3 — encode the algebra as RDF

This is the actual "get access to the RDF" step. `query_to_rdf()` walks `parsed.algebra` and encodes it into an `rdflib.Graph` using the `salg:` ontology — one triple pattern, filter, projection, etc. per algebra node. `root` is the graph node standing in for the query's top-level operator, needed to decode the graph back into a `Query` later.

In [4]:
graph, root = starsparql.query_to_rdf(parsed)
print("encoded triples:", len(graph))

encoded triples: 143


## Step 4 — edit the query by editing the RDF

Rewrite the triple pattern's predicate directly in the RDF graph — `ex:knows` → `ex:likes`. No text-level find/replace anywhere; this edits the query's own structure.

In [5]:
from starsparql.vocab import SALG

for s, p, o in list(graph.triples((None, SALG.predicate, EX.knows))):
    graph.remove((s, p, o))
    graph.add((s, p, EX.likes))

## Step 5 — decode back into a real `Query`, and run it

`rdf_to_query()` turns the edited RDF back into an executable `Query`. Running it against the *same* graph from Step 1 proves the edit is semantic — the result set genuinely changes, from `(bob, carol)` to `(bob, dana)`.

In [6]:
modified_query = starsparql.rdf_to_query(graph, root)

for row in g.query(modified_query):
    print(g.qname(row.s), g.qname(row.o))

ex:bob ex:dana


## Step 6 — export the modified query back to text

`translate_algebra_12()` walks a `Query`'s `.algebra` straight to SPARQL 1.2 text — no intermediate RDF round-trip needed when text is all you want.

In [7]:
print(starsparql.translate_algebra_12(modified_query))

SELECT ?s ?o{?s <http://example.org/likes> ?o. }


## Step 7 — load the ontology and shapes graphs

`starsparql.ontology_graph()` and `starsparql.shapes_graph()` each return a fresh **plain `rdflib.Graph`**, not a `StarLayerGraph` — confirmed from source (`starsparql/ontology/__init__.py`, `starsparql/ontology/sparql_shapes.py`). They contain the `salg:`/SHACL *vocabulary itself* (classes, properties, shape definitions) parsed with plain `format="turtle"`, never actual triple-term or reification data, so there's nothing RDF-1.2-specific for them to carry — a plain `rdflib.Graph` is the right and complete return type here. `shapes_graph()` additionally requires `pyshacl` to be installed.

In [8]:
ontology = starsparql.ontology_graph()
print("ontology triples:", len(ontology))

shapes = starsparql.shapes_graph()
print("shapes triples:", len(shapes))

ontology triples: 333
shapes triples: 1265


## Step 8 — validate an encoded query, with RDFS reasoning over the ontology

`starsparql.validate()` runs the shapes graph against a data graph with `inference="rdfs"` and `ont_graph=ontology_graph()` — real RDFS reasoning before SHACL validation, which is what lets shapes like `GraphPatternShape`/`ExpressionShape` check a single abstract superclass (`salg:Expression`) instead of enumerating every concrete operator/builtin by name. This example builds its query with plain rdflib's own `prepareQuery()` (not `starsparql.prepare_query_12()`) deliberately, to show `validate()` works on any correctly-encoded `salg:` graph, regardless of what produced it.

In [9]:
from rdflib.plugins.sparql import prepareQuery

q = prepareQuery("SELECT ?s ?o WHERE { ?s ex:knows ?o }", initNs={"ex": EX})
encoded, root = starsparql.query_to_rdf(q)
print("encoded query triples:", len(encoded))

conforms, results_graph, results_text = starsparql.validate(encoded)
print("conforms:", conforms)

encoded query triples: 143
conforms: True


## Further work

- **`sh:sparql` and `sh:SPARQLRule`** use these same underlying query functions from inside SHACL shapes — see the [SHACL inference rules guide](04b-shacl-inference-rules.ipynb) and the [SHACL and SPARQL guide](04c-shacl-and-sparql.ipynb).